# Capitolo 11 — Dimensione dell'input (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap11_dimensione_input_java.ipynb)

Questo notebook accompagna il capitolo 11 «Dimensione dell'input» delle dispense (volume 4, *Analisi e ricorsione*) e della [pagina del sito](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/11-dimensione-input/). Contiene il codice **Java** degli algoritmi con pseudocodice del capitolo: `SumArray`. Prima di `SumArray`, una sezione calcola la dimensione dell'input $|I|$ (numero di bit) degli esempi del capitolo. Le righe dello pseudocodice sono numerate come nelle dispense (la riga 1 è l'intestazione `function`) e un array `cnt` conta quante volte viene eseguita ciascuna riga.

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime ▸ Esegui tutto* (oppure cella per cella con Maiusc+Invio). Non serve installare nulla.

Ogni programma è una cella `%%writefile` che scrive il file sorgente `.java`, seguita da una cella che lo compila con `javac` e lo esegue con `java`. La prima cella verifica (o installa) il JDK.

**Indici.** Nello pseudocodice gli array sono indicizzati da 1; nel codice Java da 0: l'elemento `a[i]` dello pseudocodice è `a[i-1]` nel codice (e un ciclo `while i <= n` da 1 diventa `while i < n` da 0). Le tabelle stampate usano gli indici dello pseudocodice.

## Preparazione: il JDK

Su Colab `java` c'è già, ma serve anche il compilatore `javac`: la cella lo verifica e, se manca, installa il JDK.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## 1. La dimensione dell'input degli esempi

**Input:** un numero naturale, due numeri naturali o un array di numeri naturali. **Output:** la dimensione dell'input $|I|$, cioè il numero di bit necessari per scriverlo in binario ($\lfloor \log_2 n \rfloor + 1$ bit per un numero $n \ge 1$).

Si veda: sezioni 11.1, 11.3, 11.4, 11.5 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/11-dimensione-input/).

In [2]:
%%writefile InputSize.java
public class InputSize
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

// "[a[from], ..., a[to-1]]"
static String fmt(int[] a, int from, int to)
{
    StringBuilder s = new StringBuilder("[");
    for (int i = from; i < to; i++)
    {
        if (i > from) s.append(", ");
        s.append(a[i]);
    }
    return s.append("]").toString();
}

    // Numero di bit di un naturale n >= 1: floor(log2 n) + 1
    static int bits(int n)
    {
        int k = 0;
        while (n > 0)
        {
            n = n / 2;        // si dimezza fino ad arrivare a 0
            k = k + 1;
        }
        return k;
    }

    // Cifre di n in base 2 (come stringa)
    static String binary(int n)
    {
        String s = "";
        while (n > 0)
        {
            s = (n % 2) + s;
            n = n / 2;
        }
        return s;
    }

    static double log2(double x)
    {
        return Math.log(x) / Math.log(2);
    }

    public static void main(String[] args)
    {
        // Esempio delle dispense: il numero di bit di alcune istanze
        int n = 37, m = 200;
        System.out.printf("Un numero: n = %d = (%s)_2, |I| = %d\n", n, binary(n), bits(n));
        System.out.printf("Due numeri: n = %d, m = %d = (%s)_2, |I| = %d + %d = %d\n", n, m, binary(m), bits(n), bits(m), bits(n) + bits(m));
        int[] a = {6, 1, 3, 8};
        int[] b = new int[4];
        int sum = 0, M = 0;
        for (int i = 0; i < 4; i++)
        {
            b[i] = bits(a[i]);
            sum += b[i];
            if (a[i] > M) M = a[i];
        }
        System.out.printf("Un array: a = %s, bit per voce = %s, |I| = %d\n", fmt(a, 0, 4), fmt(b, 0, 4), sum);

        // I limiti delle dispense (valori arrotondati a due decimali)
        System.out.printf("\n");
        System.out.printf("Un numero: log2(n) = %.2f <= |I| = %d <= 2 log2(n) = %.2f\n", log2(n), bits(n), 2 * log2(n));
        double s = log2(n) + log2(m);
        System.out.printf("Due numeri: log2(n) + log2(m) = %.2f <= |I| = %d <= 2 (log2(n) + log2(m)) = %.2f\n", s, bits(n) + bits(m), 2 * s);
        int mx = Math.max(n, m);
        System.out.printf("Due numeri e massimo: log2(max) = %.2f <= |I| = %d <= 4 log2(max) = %.2f\n", log2(mx), bits(n) + bits(m), 4 * log2(mx));
        System.out.printf("Un array: n = %d <= |I| = %d <= n (1 + log2 M) = %.2f\n", 4, sum, 4 * (1 + log2(M)));

        // Il limite superiore non risulta stretto: a[1] = M = 2^n, a[2..n] = 1 ha |I| = 2n, ma n(1 + log2 M) = n(n + 1)
        System.out.printf("\n");
        System.out.printf("   n |      M |  |I| = 2n | n (1 + log2 M)\n");
        for (int k = 1; k <= 10; k++)
        {
            int big = 1 << k;
            int size = bits(big) + (k - 1);     // a[1] = M, a[2..n] = 1 (1 bit each)
            System.out.printf("%4d | %6d | %9d | %13d\n", k, big, size, k * (k + 1));
        }
    }
}

Writing InputSize.java


In [3]:
!javac InputSize.java && java InputSize

Un numero: n = 37 = (100101)_2, |I| = 6
Due numeri: n = 37, m = 200 = (11001000)_2, |I| = 6 + 8 = 14
Un array: a = [6, 1, 3, 8], bit per voce = [3, 1, 2, 4], |I| = 10

Un numero: log2(n) = 5.21 <= |I| = 6 <= 2 log2(n) = 10.42
Due numeri: log2(n) + log2(m) = 12.85 <= |I| = 14 <= 2 (log2(n) + log2(m)) = 25.71
Due numeri e massimo: log2(max) = 7.64 <= |I| = 14 <= 4 log2(max) = 30.58
Un array: n = 4 <= |I| = 10 <= n (1 + log2 M) = 16.00

   n |      M |  |I| = 2n | n (1 + log2 M)
   1 |      2 |         2 |             2
   2 |      4 |         4 |             6
   3 |      8 |         6 |            12
   4 |     16 |         8 |            20
   5 |     32 |        10 |            30
   6 |     64 |        12 |            42
   7 |    128 |        14 |            56
   8 |    256 |        16 |            72
   9 |    512 |        18 |            90
  10 |   1024 |        20 |           110


**Prova tu.** Cambia il numero $n$ e riesegui la cella.

In [4]:
%%writefile ProvaInputSize.java
public class ProvaInputSize
{
    public static void main(String[] args)
    {
        int n = 1000;        // <-- cambia il numero
        System.out.printf("n = %d = (%s)_2, |I| = %d, log2(n) = %.2f\n", n, InputSize.binary(n), InputSize.bits(n), InputSize.log2(n));
    }
}

Writing ProvaInputSize.java


In [5]:
!javac ProvaInputSize.java && java ProvaInputSize

n = 1000 = (1111101000)_2, |I| = 10, log2(n) = 9.97


## 2. Somma delle componenti di un array: SumArray

**Input:** un array $a[1..n]$ di $n \ge 1$ numeri naturali. **Output:** $a[1]+a[2]+\cdots+a[n]$. Numero di passi: $S(n) = 3n + 4$, qualunque siano i valori.

Pseudocodice: sezione 11.6 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/11-dimensione-input/).

In [6]:
%%writefile SumArray.java
public class SumArray
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

// "[a[from], ..., a[to-1]]"
static String fmt(int[] a, int from, int to)
{
    StringBuilder s = new StringBuilder("[");
    for (int i = from; i < to; i++)
    {
        if (i > from) s.append(", ");
        s.append(a[i]);
    }
    return s.append("]").toString();
}

    // SumArray(a, n)
    static int sumArray(int[] a)
    {
        int n = a.length;
        int s = 0; cnt[2]++;
        int i = 0; cnt[3]++;               // i := 1 nello pseudocodice
        cnt[4]++;                          // la condizione del while si conta a ogni valutazione
        while (i < n)                      // i <= n nello pseudocodice
        {
            s = s + a[i]; cnt[5]++;
            i = i + 1; cnt[6]++;
            cnt[4]++;
        }
        cnt[8]++;
        return s;
    }

    public static void main(String[] args)
    {
        int[] a = {6, 1, 3, 8};
        int n = a.length;
        reset();
        int s = sumArray(a);
        System.out.printf("a = %s, n = %d, somma = %d\n", fmt(a, 0, n), n, s);
        printSteps(2, 3, 4, 5, 6, 8);
        System.out.printf("Formula: S(n) = 3n + 4 = %d\n", 3 * n + 4);

        // Caso peggiore: l'array di |I| uni ha dimensione dell'input |I| e richiede esattamente 3|I| + 4 passi
        System.out.printf("\n");
        System.out.printf("Array di |I| uni:\n");
        int[] sizes = {4, 8, 16, 32};
        for (int size : sizes)
        {
            int[] ones = new int[size];
            java.util.Arrays.fill(ones, 1);
            reset();
            sumArray(ones);
            System.out.printf("|I| = %2d: S = %3d = 3|I| + 4 = %d\n", size, total(), 3 * size + 4);
        }
    }
}

Writing SumArray.java


In [7]:
!javac SumArray.java && java SumArray

a = [6, 1, 3, 8], n = 4, somma = 18
Riga | Esecuzioni
   2 |          1
   3 |          1
   4 |          5
   5 |          4
   6 |          4
   8 |          1
Passi totali S = 16
Formula: S(n) = 3n + 4 = 16

Array di |I| uni:
|I| =  4: S =  16 = 3|I| + 4 = 16
|I| =  8: S =  28 = 3|I| + 4 = 28
|I| = 16: S =  52 = 3|I| + 4 = 52
|I| = 32: S = 100 = 3|I| + 4 = 100


**Prova tu.** Cambia l'array e verifica che i passi siano sempre $3n + 4$.

In [8]:
%%writefile ProvaSumArray.java
public class ProvaSumArray
{
    public static void main(String[] args)
    {
        int[] a = {5, 1, 9, 2, 7};      // <-- cambia l'array
        SumArray.reset();
        int s = SumArray.sumArray(a);
        System.out.printf("somma = %d, S = %d, 3n + 4 = %d\n", s, SumArray.total(), 3 * a.length + 4);
    }
}

Writing ProvaSumArray.java


In [9]:
!javac ProvaSumArray.java && java ProvaSumArray

somma = 24, S = 19, 3n + 4 = 19
